# Seizure Anticipation: BIDS & Wearable Data Integration Workflow

This notebook integrates both BIDS-formatted (ds005873) and Wearable Device (Oregon) datasets for epilepsy seizure anticipation in the Epilepsee-AI project.

**Workflow Steps:**
1. Set up environment and imports
2. Load and validate BIDS dataset (ds005873)
3. Load and validate Wearable dataset (Oregon)
4. Extract features and labels from both sources
5. Create unified PyTorch datasets
6. Persist processed data for training

## Section 1: Set Up Notebook Environment and Project Imports

In [ ]:
# Add project root to sys.path
import sys
from pathlib import Path

project_root = Path('/home/tnzr/Documents/FIU/Research/Epilepsee-AI')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path updated: {sys.path[0]}")

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import logging
from datetime import datetime
import json
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s - %(levelname)s - %(message)s'
)

print("✓ Core packages imported")

In [ ]:
# Import from Epilepsee-AI project
from config.config import DEFAULT_CONFIG, DataConfig
from src.data_loader import (
    BIDSDataLoader,
    WearableDeviceDataLoader,
    SeizureDataset,
    augment_preictal_sample
)

print("✓ Epilepsee-AI modules imported")
print(f"\nDefault config dataset root: {DEFAULT_CONFIG.data.dataset_root}")

## Section 2: Load `DataConfig` and Override Dataset Paths

In [ ]:
# Create custom config with specific dataset paths
from dataclasses import replace

# BIDS dataset path
bids_root = "/media/tnzr/HDD11/Datasets/ds005873"
wearable_root = "/media/tnzr/HDD11/Datasets/WearablwDevice-Oregon"

# Verify paths exist
bids_path = Path(bids_root)
wearable_path = Path(wearable_root)

print(f"BIDS dataset exists: {bids_path.exists()}")
print(f"Wearable dataset exists: {wearable_path.exists()}")

# Create config for BIDS data loader
bids_config = replace(DEFAULT_CONFIG.data, dataset_root=bids_root)
print(f"\n✓ BIDS config created: {bids_root}")

## Section 3: Validate BIDS (`ds005873`) Availability and Subject Enumeration

In [ ]:
# Initialize BIDS data loader
try:
    bids_loader = BIDSDataLoader(bids_config)
    print("✓ BIDS DataLoader initialized successfully")
    
    # Get all subjects
    bids_subjects = bids_loader.get_subjects()
    print(f"\nFound {len(bids_subjects)} BIDS subjects")
    print(f"Subject IDs (first 10): {bids_subjects[:10]}")
    if len(bids_subjects) > 10:
        print(f"Subject IDs (last 10): {bids_subjects[-10:]}")
        
except Exception as e:
    print(f"✗ Failed to initialize BIDS loader: {e}")

## Section 4: Scan BIDS Events and Build Recording-Level Seizure Summary

In [ ]:
# Get all BIDS recordings
try:
    bids_recordings = bids_loader.list_all_recordings()
    print(f"Found {len(bids_recordings)} BIDS recordings")
    
    # Convert to DataFrame for analysis
    bids_df = pd.DataFrame(bids_recordings)
    
    print(f"\nBIDS Recordings Summary:")
    print(f"  Recordings with seizures: {(bids_df['num_seizures'] > 0).sum()}")
    print(f"  Total seizure events: {bids_df['num_seizures'].sum()}")
    print(f"  Unique subjects: {bids_df['subject_id'].nunique()}")
    print(f"  Unique sessions: {bids_df['session_id'].nunique()}")
    
    print(f"\nSample recordings:")
    print(bids_df.head())
    
except Exception as e:
    print(f"✗ Error listing BIDS recordings: {e}")

In [ ]:
# Visualize seizure distribution
if len(bids_recordings) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Seizure count per recording
    bids_df['num_seizures'].value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color='steelblue'
    )
    axes[0].set_title('BIDS: Seizure Count Distribution')
    axes[0].set_xlabel('Number of Seizures')
    axes[0].set_ylabel('Recording Count')
    
    # Seizures per subject
    seizures_per_subject = bids_df.groupby('subject_id')['num_seizures'].sum()
    seizures_per_subject.plot(kind='hist', ax=axes[1], bins=20, color='coral')
    axes[1].set_title('BIDS: Seizures per Subject')
    axes[1].set_xlabel('Seizure Count')
    axes[1].set_ylabel('Subject Count')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Seizure statistics per subject:")
    print(seizures_per_subject.describe())

## Section 5: Load a Sample BIDS Run (EDF + `events.tsv`) and Inspect Signals

In [ ]:
# Load sample BIDS recording with seizures
try:
    seizure_recordings_bids = bids_df[bids_df['num_seizures'] > 0]
    
    if len(seizure_recordings_bids) > 0:
        sample_rec = seizure_recordings_bids.iloc[0]
        print(f"Loading sample BIDS recording:")
        print(f"  Subject: {sample_rec['subject_id']}")
        print(f"  Session: {sample_rec['session_id']}")
        print(f"  Run: {sample_rec['run_id']}")
        print(f"  Seizures: {sample_rec['num_seizures']}")
        
        # Try to load signals (may require MNE)
        try:
            signals = bids_loader.load_subject_session(
                sample_rec['subject_id'],
                sample_rec['session_id'],
                sample_rec['run_id']
            )
            
            print(f"\nLoaded signal modalities:")
            for modality, data in signals.items():
                if modality != 'events':
                    sig, fs = data
                    print(f"  {modality}: shape={sig.shape}, fs={fs} Hz")
                    
        except ImportError:
            print("Note: MNE-Python not installed. Skip EDF loading.")
    else:
        print("No BIDS recordings with seizures found")
        
except Exception as e:
    print(f"Error loading sample BIDS data: {e}")

## Section 6: Initialize Wearable Loader and Parse Master Seizure Database

In [ ]:
# Initialize Wearable device loader
try:
    wearable_loader = WearableDeviceDataLoader(
        DEFAULT_CONFIG.data,
        dataset_root=wearable_root
    )
    print("✓ Wearable DataLoader initialized successfully")
    
    # Inspect master database columns
    print(f"\nMaster database columns:")
    print(f"  Total rows: {len(wearable_loader.seizure_db)}")
    print(f"  Columns: {list(wearable_loader.seizure_db.columns)}")
    
except Exception as e:
    print(f"✗ Failed to initialize wearable loader: {e}")

## Section 7: Enumerate Wearable Recordings and Check Directory Matching

In [ ]:
# Get all wearable recordings
try:
    wearable_recordings = wearable_loader.get_all_recordings()
    print(f"Found {len(wearable_recordings)} wearable recordings")
    
    # Convert to DataFrame
    wearable_df = pd.DataFrame(wearable_recordings)
    
    print(f"\nWearable Recordings Summary:")
    print(f"  Recordings with seizures: {(wearable_df['has_seizure']).sum()}")
    print(f"  Total seizure events: {wearable_df['num_seizures'].sum()}")
    print(f"  Unique subjects: {wearable_df['subject_id'].nunique()}")
    print(f"  Unique watches: {wearable_df['watch_id'].nunique()}")
    print(f"  Recordings with found directory: {(wearable_df['recording_dir'].notna()).sum()}")
    
    print(f"\nSample wearable recordings:")
    display(wearable_df[['subject_id', 'watch_id', 'has_seizure', 'num_seizures', 'recording_dir']].head())
    
except Exception as e:
    print(f"✗ Error listing wearable recordings: {e}")

## Section 8: Extract Windowed Wearable Features and Countdown Labels

In [ ]:
# Extract features from wearable recordings with seizures
all_wearable_features = []
all_wearable_labels = []
all_wearable_subject_ids = []
all_wearable_sample_times = []

seizure_recordings_wearable = wearable_df[wearable_df['has_seizure'] & (wearable_df['recording_dir'].notna())]
print(f"Processing {len(seizure_recordings_wearable)} wearable recordings with seizures...\n")

for idx, rec_dict in seizure_recordings_wearable.head(3).iterrows():  # Process first 3 for demo
    rec = rec_dict.to_dict()
    print(f"Processing: {rec['subject_id']}/{rec['watch_id']}")
    print(f"  Seizure times: {rec['seizure_times']}")
    
    try:
        features, labels, times = wearable_loader.extract_features_from_recording(
            rec,
            window_s=60.0
        )
        
        if features is not None:
            print(f"  ✓ Features: {features.shape}")
            print(f"    Preictal: {(labels >= 0).sum()}, Interictal: {(labels < 0).sum()}")
            
            all_wearable_features.append(features)
            all_wearable_labels.append(labels)
            all_wearable_subject_ids.append(np.repeat(rec['subject_id'], len(labels)))
            all_wearable_sample_times.append(times)
        else:
            print(f"  ✗ No features extracted")
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    print()

## Section 9: Construct `SeizureDataset` and Inspect Weights/Class Balance

In [ ]:
# Create SeizureDataset from wearable features
if all_wearable_features:
    # Concatenate features
    X = np.concatenate(all_wearable_features)  # (N, T, F)
    y = np.concatenate(all_wearable_labels)    # (N,)
    subject_ids = np.concatenate(all_wearable_subject_ids)  # (N,)
    
    print(f"Combined features shape: {X.shape}")
    print(f"Labels shape: {y.shape}")
    print(f"Subject IDs shape: {subject_ids.shape}")
    
    # Create dataset
    dataset = SeizureDataset(
        features=X,
        labels=y,
        subject_ids=subject_ids
    )
    
    print(f"\n✓ SeizureDataset created with {len(dataset)} samples")
    print(f"\nClass distribution:")
    print(dataset.class_distribution)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Label distribution
    axes[0].hist(y[y >= 0], bins=30, alpha=0.7, label='Preictal', color='red')
    axes[0].hist(y[y < 0], bins=30, alpha=0.7, label='Interictal', color='blue')
    axes[0].set_xlabel('Countdown (minutes)')
    axes[0].set_ylabel('Sample Count')
    axes[0].set_title('Label Distribution')
    axes[0].legend()
    
    # Sample weights
    axes[1].hist(dataset.weights, bins=30, color='green', alpha=0.7)
    axes[1].set_xlabel('Sample Weight')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Temporal Importance Weights')
    
    plt.tight_layout()
    plt.show()
else:
    print("No features extracted to create dataset")

## Section 10: Run Preictal Augmentation Sanity Checks

In [ ]:
# Test augmentation on preictal samples
if all_wearable_features:
    preictal_mask = y >= 0
    preictal_indices = np.where(preictal_mask)[0]
    
    print(f"Testing augmentation on preictal samples...")
    print(f"Found {len(preictal_indices)} preictal samples\n")
    
    if len(preictal_indices) > 0:
        # Test on first preictal sample
        sample_idx = preictal_indices[0]
        sample_features = X[sample_idx]
        sample_label = y[sample_idx]
        
        print(f"Original sample:")
        print(f"  Shape: {sample_features.shape}")
        print(f"  Label (min to seizure): {sample_label:.2f} min")
        print(f"  Mean value: {sample_features.mean():.4f}")
        
        # Augment
        rng = np.random.default_rng(42)
        augmented = augment_preictal_sample(
            sample_features,
            sample_label,
            DEFAULT_CONFIG.data,
            rng
        )
        
        print(f"\nAugmentation results:")
        print(f"  Generated {len(augmented)} augmented versions")
        
        for i, (aug_feat, aug_label) in enumerate(augmented[:3]):
            print(f"  Aug {i}: shape={aug_feat.shape}, label={aug_label:.2f}, mean={aug_feat.mean():.4f}")
            assert aug_feat.shape == sample_features.shape, "Shape mismatch!"
            assert aug_label == sample_label, "Label mismatch!"
        
        print(f"\n✓ Augmentation sanity checks passed")

## Section 11: Persist Debug Artifacts (NPZ/CSV/JSON) for Training Pipeline

In [ ]:
# Save processed data for training pipeline
output_dir = Path('/home/tnzr/Documents/FIU/Research/Epilepsee-AI/data')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if all_wearable_features:
    # Save feature tensors
    features_file = output_dir / f"wearable_features_{timestamp}.npz"
    np.savez_compressed(
        features_file,
        features=X,
        labels=y,
        subject_ids=subject_ids
    )
    print(f"✓ Saved features to {features_file}")
    
    # Save recording metadata
    metadata_file = output_dir / f"wearable_metadata_{timestamp}.csv"
    wearable_df.to_csv(metadata_file, index=False)
    print(f"✓ Saved metadata to {metadata_file}")
    
    # Save config snapshot
    config_snapshot = {
        'timestamp': timestamp,
        'dataset': 'wearable',
        'num_samples': len(X),
        'feature_shape': X.shape,
        'preictal_samples': int((y >= 0).sum()),
        'interictal_samples': int((y < 0).sum()),
        'class_distribution': {
            'preictal': int((y >= 0).sum()),
            'interictal': int((y < 0).sum())
        },
        'config': {
            'feature_window_s': float(DEFAULT_CONFIG.data.feature_window_s),
            'feature_step_s': float(DEFAULT_CONFIG.data.feature_step_s),
            'pre_ictal_window_s': float(DEFAULT_CONFIG.data.pre_ictal_window_s)
        }
    }
    
    config_file = output_dir / f"config_snapshot_{timestamp}.json"
    with open(config_file, 'w') as f:
        json.dump(config_snapshot, f, indent=2)
    print(f"✓ Saved config snapshot to {config_file}")
    
    print(f"\n✓ All artifacts saved to {output_dir}")

In [ ]:
# Final summary
print("\n" + "="*70)
print("INTEGRATION WORKFLOW SUMMARY")
print("="*70)

print(f"\nBIDS Dataset (ds005873):")
print(f"  ✓ Subjects: {len(bids_subjects)}")
print(f"  ✓ Recordings: {len(bids_recordings)}")
print(f"  ✓ Seizure recordings: {(bids_df['num_seizures'] > 0).sum()}")
print(f"  ✓ Total seizures: {bids_df['num_seizures'].sum()}")

print(f"\nWearable Dataset (Oregon):")
print(f"  ✓ Recordings: {len(wearable_recordings)}")
print(f"  ✓ Seizure recordings: {wearable_df['has_seizure'].sum()}")
print(f"  ✓ Total seizures: {wearable_df['num_seizures'].sum()}")
print(f"  ✓ Records with found directory: {(wearable_df['recording_dir'].notna()).sum()}")

if all_wearable_features:
    print(f"\nProcessed Wearable Data:")
    print(f"  ✓ Samples extracted: {len(X)}")
    print(f"  ✓ Feature shape: {X.shape}")
    print(f"  ✓ Preictal: {(y >= 0).sum()} | Interictal: {(y < 0).sum()}")
    print(f"  ✓ Class imbalance ratio: {(y >= 0).sum() / (y < 0).sum():.3f}")

print(f"\n✓ Integration workflow completed successfully!")
print("="*70)